# REIT Sentiment Analysis Module



## INSTALL DEPENDENCIES

In [ ]:
!pip install pandas numpy requests beautifulsoup4 transformers torch accelerate lxml newspaper4k matplotlib seaborn

# Note: newspaper4k is the maintained fork of newspaper3k, compatible with Python 3.10+
# newspaper3k is broken on Python 3.10+ due to dropped lxml_html_clean dependency

## IMPORTS

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from typing import List, Dict, Tuple, Optional
import time
import re
import hashlib
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Optional imports - degrade if not available
try:
    from newspaper import Article as NewspaperArticle
    HAS_NEWSPAPER = True
except ImportError:
    HAS_NEWSPAPER = False
    print("⚠ newspaper4k not installed. Article body fetching disabled.")
    print("  Install with: pip install newspaper4k")

try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import seaborn as sns
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False
    print("⚠ matplotlib/seaborn not installed. Visualization disabled.")

## CONFIGURATION

In [ ]:
class AnalysisConfig:

    """Central configuration for the analysis pipeline."""

    # API Keys
    ALPHA_VANTAGE_KEY: Optional[str] = None
    NEWSDATA_KEY: Optional[str] = None
    MARKETAUX_KEY: Optional[str] = None

    PRIMARY_MODEL = "ProsusAI/finbert"
    SECONDARY_MODEL = "yiyanghkust/finbert-tone"  # Set to None to skip ensemble
    ENSEMBLE_WEIGHT_PRIMARY = 0.6   

    # --- Scoring Weights ---
    MACRO_WEIGHT = 0.4
    MICRO_WEIGHT = 0.6
    RECENCY_DECAY = 0.1         # Exponential decay factor for temporal weighting
    SOURCE_QUALITY_BOOST = 1.3  # Multiplier for high-quality sources

    # --- Source Quality Tiers ---
    TIER1_SOURCES = {
        'reuters', 'bloomberg', 'the straits times', 'business times',
        'the edge singapore', 'channel newsasia', 'cnbc', 'financial times',
        'wall street journal', 'the business times', 'sgx', 'mas',
        'seeking alpha', 'the edge markets', 'nikkei asia'
    }
    TIER2_SOURCES = {
        'yahoo finance', 'marketwatch', 'investing.com', 'morningstar',
        'the motley fool', 'benzinga', 'barrons', 'reit.com',
        'property guru', 'edgeprop', 'mingtiandi'
    }

    # --- Geography Configuration ---
    # Set per REIT based on where its properties are located
    # Options: 'SG', 'US', 'UK', 'EU', 'GLOBAL'
    GEOGRAPHIES: list = None  # Will default to ['SG'] if not set

    # --- Extra Relevance Keywords ---
    # Add REIT-specific terms here (e.g., 'grocery anchored' for ODBU)
    EXTRA_RELEVANCE_KEYWORDS: list = None

    # --- Base Relevance Keywords (always active) ---
    REIT_RELEVANCE_KEYWORDS = [
        'reit', 'real estate', 'property', 'rental', 'tenant', 'occupancy',
        'distribution', 'dpu', 'yield', 'gearing', 'leverage',
        'npi', 'net property income', 'cap rate', 'valuation', 'acquisition',
        'divestment', 'rights issue', 'placement', 'refinancing',
        'interest rate', 'inflation', 'gdp', 'economy', 'monetary policy',
        'fed', 'treasury', 'bond', 'credit',
        'office', 'retail', 'industrial', 'logistics', 'data centre',
        'hospitality', 'healthcare', 'commercial', 'residential',
        'sgx', 's-reit',
    ]

    # --- Geography-based Macro Query Templates ---
    GEO_MACRO_QUERIES = {
        'SG': [
            "Singapore GDP growth",
            "Singapore economy outlook",
            "MAS monetary policy",
            "Singapore interest rates SORA",
            "Singapore inflation CPI",
            "Singapore property market outlook",
            "Singapore office vacancy rate",
            "Singapore retail rental market",
            "Singapore industrial property demand",
            "Singapore REIT sector outlook",
            "S-REIT distribution yield spread",
            "Singapore commercial property transaction",
        ],
        'US': [
            "US GDP growth outlook",
            "US Federal Reserve interest rates",
            "US inflation CPI",
            "US treasury yield",
            "US commercial real estate outlook",
            "US retail property vacancy rate",
            "US self storage market demand",
            "US grocery anchored retail property",
            "US cap rate commercial real estate",
            "US REIT sector performance",
            "US consumer spending retail",
            "US employment jobs data",
        ],
        'UK': [
            "UK GDP growth outlook",
            "Bank of England interest rates",
            "UK inflation CPI",
            "UK gilt yield",
            "UK commercial property market outlook",
            "UK office vacancy rate",
            "UK government property lease",
            "UK real estate investment outlook",
            "UK cap rate commercial property",
            "UK REIT sector performance",
            "UK rental market outlook",
            "UK economy outlook",
        ],
        'EU': [
            "European GDP growth outlook",
            "ECB interest rates policy",
            "Eurozone inflation CPI",
            "European commercial property outlook",
            "Germany office property market",
            "France commercial real estate",
            "Spain retail property market",
            "European REIT sector outlook",
            "European cap rate office retail",
            "Eurozone economy outlook",
            "Germany GDP economic outlook",
            "European rental yield outlook",
        ],
        'GLOBAL': [
            "global interest rate outlook",
            "global real estate investment outlook",
            "Asia Pacific real estate investment",
            "global REIT performance comparison",
        ],
    }


## UTILITY FUNCTIONS

In [ ]:
def clean_html(text: str) -> str:
    
    if not text:
        return ""
    # Remove HTML tags
    clean = re.sub(r'<[^>]+>', '', text)
    # Decode common HTML entities
    clean = clean.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    clean = clean.replace('&#39;', "'").replace('&quot;', '"')
    # Remove excessive whitespace
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean


def normalize_title(title: str) -> str:
    
    # Lowercase, remove punctuation, collapse whitespace
    t = title.lower()
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def deduplicate_articles(articles: List[Dict], similarity_threshold: float = 0.8) -> List[Dict]:
 
    if not articles:
        return []

    seen_titles = {}
    unique_articles = []

    for article in articles:
        norm_title = normalize_title(article.get('title', ''))
        if not norm_title or len(norm_title) < 10:
            continue

        # Check against all seen titles using simple word overlap
        is_duplicate = False
        title_words = set(norm_title.split())

        for seen_norm, seen_idx in seen_titles.items():
            seen_words = set(seen_norm.split())
            if not title_words or not seen_words:
                continue
            # Jaccard similarity
            intersection = len(title_words & seen_words)
            union = len(title_words | seen_words)
            similarity = intersection / union if union > 0 else 0

            if similarity >= similarity_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            seen_titles[norm_title] = len(unique_articles)
            unique_articles.append(article)

    return unique_articles


def check_relevance(article: Dict, reit_name: str, config: AnalysisConfig) -> bool:

    text = f"{article.get('title', '')} {article.get('description', '')}".lower()
    reit_lower = reit_name.lower()

    # Always include if it mentions the REIT by name
    if reit_lower in text or any(part in text for part in reit_lower.split() if len(part) > 3):
        return True

    # Build combined keyword list (base + extra)
    all_keywords = list(config.REIT_RELEVANCE_KEYWORDS)
    if config.EXTRA_RELEVANCE_KEYWORDS:
        all_keywords.extend(config.EXTRA_RELEVANCE_KEYWORDS)

    # For macro articles, check against combined relevance keywords
    if article.get('category') == 'macro':
        return any(kw in text for kw in all_keywords)

    # For micro articles, must mention the REIT or sector-specific terms
    reit_keywords = ['reit', 's-reit', 'real estate investment trust', reit_lower,
                     'capitaland', 'mapletree', 'ascendas', 'frasers', 'keppel',
                     'sgx', 'distribution per unit', 'dpu']
    if config.EXTRA_RELEVANCE_KEYWORDS:
        reit_keywords.extend(config.EXTRA_RELEVANCE_KEYWORDS)
    return any(kw in text for kw in reit_keywords)


def get_source_quality_multiplier(source: str, config: AnalysisConfig) -> float:

    source_lower = source.lower().strip()
    for tier1 in config.TIER1_SOURCES:
        if tier1 in source_lower:
            return config.SOURCE_QUALITY_BOOST
    for tier2 in config.TIER2_SOURCES:
        if tier2 in source_lower:
            return (1.0 + config.SOURCE_QUALITY_BOOST) / 2  # Moderate boost
    return 1.0


def calculate_recency_weight(pub_date_str: str, reference_date: datetime,
                              decay: float = 0.1) -> float:

    try:
        # Parse various date formats from different sources
        for fmt in ['%a, %d %b %Y %H:%M:%S %Z', '%a, %d %b %Y %H:%M:%S GMT',
                    '%Y-%m-%dT%H:%M:%SZ', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d']:
            try:
                pub_date = datetime.strptime(pub_date_str.strip(), fmt)
                days_ago = (reference_date - pub_date).days
                days_ago = max(0, days_ago)
                return np.exp(-decay * days_ago)
            except ValueError:
                continue
    except Exception:
        pass
    return 0.5  # Default weight for unparseable dates


def fetch_article_body(url: str, timeout: int = 10) -> str:

    if not HAS_NEWSPAPER or not url:
        return ""
    try:
        article = NewspaperArticle(url)
        article.download()
        article.parse()
        # Return first 1000 chars to keep FinBERT input manageable
        return (article.text or "")[:1000]
    except Exception:
        return ""

## NEWS SOURCE CLASSES

In [ ]:
class GoogleNewsSource:

    @staticmethod
    def fetch(query: str, days_back: int = 30, region: str = "SG") -> List[Dict]:
        articles = []
        try:
            url = (f"https://news.google.com/rss/search?"
                   f"q={query}+when:{days_back}d&hl=en-{region}&gl={region}&ceid={region}:en")
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'xml')

            for item in soup.find_all('item'):
                article = {
                    'title': item.title.text.strip() if item.title else '',
                    'link': item.link.text.strip() if item.link else '',
                    'pub_date': item.pubDate.text.strip() if item.pubDate else '',
                    'source': item.source.text.strip() if item.source else 'Google News',
                    'description': clean_html(item.description.text if item.description else ''),
                    'data_source': 'google_news'
                }
                articles.append(article)
        except Exception as e:
            print(f"    ⚠ Google News error for '{query}': {e}")
        return articles


class AlphaVantageNewsSource:

    @staticmethod
    def fetch(query: str, api_key: str, limit: int = 50) -> List[Dict]:
        if not api_key:
            return []
        articles = []
        try:
            url = (f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
                   f"&tickers={query}&limit={limit}&apikey={api_key}")
            response = requests.get(url, timeout=15)
            data = response.json()

            for item in data.get('feed', []):
                article = {
                    'title': item.get('title', ''),
                    'link': item.get('url', ''),
                    'pub_date': item.get('time_published', ''),
                    'source': item.get('source', 'Alpha Vantage'),
                    'description': item.get('summary', ''),
                    'data_source': 'alpha_vantage',
                    # Alpha Vantage provides its own sentiment - useful for comparison
                    'av_sentiment': item.get('overall_sentiment_score', None),
                    'av_label': item.get('overall_sentiment_label', None)
                }
                articles.append(article)
        except Exception as e:
            print(f"    ⚠ Alpha Vantage error: {e}")
        return articles

class NewsDataSource:

    @staticmethod
    def fetch(query: str, api_key: str, country: str = "sg", days_back: int = 7) -> List[Dict]:
        if not api_key:
            return []
        articles = []
        try:
            url = (f"https://newsdata.io/api/1/news?"
                   f"apikey={api_key}&q={query}&country={country}"
                   f"&language=en&category=business")
            response = requests.get(url, timeout=15)
            data = response.json()

            for item in data.get('results', []):
                article = {
                    'title': item.get('title', ''),
                    'link': item.get('link', ''),
                    'pub_date': item.get('pubDate', ''),
                    'source': item.get('source_name', 'NewsData'),
                    'description': item.get('description', '') or '',
                    'data_source': 'newsdata'
                }
                articles.append(article)
        except Exception as e:
            print(f"    ⚠ NewsData error: {e}")
        return articles


class MarketauxNewsSource:

    @staticmethod
    def fetch(query: str, api_key: str, country: str = "sg",
              days_back: int = 7, limit: int = 50) -> List[Dict]:
        if not api_key:
            return []
        articles = []
        try:
            date_from = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
            url = (f"https://api.marketaux.com/v1/news/all?"
                   f"api_token={api_key}&search={query}"
                   f"&countries={country}&language=en"
                   f"&published_after={date_from}&limit={limit}")
            response = requests.get(url, timeout=15)
            data = response.json()

            for item in data.get('data', []):
                # Extract entity-level sentiment if available
                entity_sentiment = None
                for entity in item.get('entities', []):
                    if entity.get('sentiment_score') is not None:
                        entity_sentiment = entity['sentiment_score']
                        break

                article = {
                    'title': item.get('title', ''),
                    'link': item.get('url', ''),
                    'pub_date': item.get('published_at', ''),
                    'source': item.get('source', 'Marketaux'),
                    'description': item.get('description', '') or '',
                    'data_source': 'marketaux',
                    'mx_entity_sentiment': entity_sentiment,
                }
                articles.append(article)
        except Exception as e:
            print(f"    ⚠ Marketaux error: {e}")
        return articles

## MAIN ANALYZER CLASS

In [ ]:
class ImprovedREITSentimentAnalyzer:


    def __init__(self, reit_name: str, reit_ticker: str = None,
                 config: AnalysisConfig = None):
        self.reit_name = reit_name
        self.reit_ticker = reit_ticker
        self.config = config or AnalysisConfig()

        # Load primary FinBERT model
        print("Loading primary FinBERT model (ProsusAI/finbert)...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.PRIMARY_MODEL)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.config.PRIMARY_MODEL)
        self.model.eval()
        print(f"✓ Primary model loaded: {self.config.PRIMARY_MODEL}")

        # Label mapping depends on model
        # ProsusAI/finbert: positive=0, negative=1, neutral=2
        self.primary_labels = ['positive', 'negative', 'neutral']

        # Load secondary model if configured (ensemble)
        self.secondary_model = None
        self.secondary_tokenizer = None
        if self.config.SECONDARY_MODEL:
            print(f"Loading secondary model ({self.config.SECONDARY_MODEL})...")
            try:
                self.secondary_tokenizer = AutoTokenizer.from_pretrained(
                    self.config.SECONDARY_MODEL)
                self.secondary_model = AutoModelForSequenceClassification.from_pretrained(
                    self.config.SECONDARY_MODEL)
                self.secondary_model.eval()
                # yiyanghkust/finbert-tone: positive=0, negative=1, neutral=2
                self.secondary_labels = ['positive', 'negative', 'neutral']
                print(f"✓ Secondary model loaded: {self.config.SECONDARY_MODEL}")
            except Exception as e:
                print(f"⚠ Could not load secondary model: {e}")
                self.secondary_model = None

    # -----------------------------------------------------------------
    # NEWS FETCHING - EXPANDED QUERIES
    # -----------------------------------------------------------------

    def get_macro_news(self, days_back: int = 30) -> List[Dict]:

        # Build query list from configured geographies
        geographies = self.config.GEOGRAPHIES or ['SG']
        macro_queries = []
        for geo in geographies:
            queries = self.config.GEO_MACRO_QUERIES.get(geo, [])
            macro_queries.extend(queries)
        # Always include global queries
        macro_queries.extend(self.config.GEO_MACRO_QUERIES.get('GLOBAL', []))
        # Deduplicate queries while preserving order
        seen = set()
        unique_queries = []
        for q in macro_queries:
            if q not in seen:
                seen.add(q)
                unique_queries.append(q)
        macro_queries = unique_queries

        print(f"  Geographies: {geographies} ({len(macro_queries)} queries)")

        all_articles = []
        for query in macro_queries:
            print(f"  {query}")
            articles = GoogleNewsSource.fetch(query, days_back)
            for a in articles:
                a['category'] = 'macro'
                a['query'] = query
            all_articles.extend(articles)
            time.sleep(0.5)

            # Also fetch from paid APIs if configured
            if self.config.NEWSDATA_KEY:
                api_articles = NewsDataSource.fetch(query, self.config.NEWSDATA_KEY,
                                                     days_back=days_back)
                for a in api_articles:
                    a['category'] = 'macro'
                    a['query'] = query
                all_articles.extend(api_articles)

        if self.config.MARKETAUX_KEY:
            geo_label = '+'.join(geographies)
            print(f"  [Marketaux] {geo_label} REIT macro")
            # Use first geography for Marketaux search term
            geo_search = {
                'SG': 'Singapore REIT property',
                'US': 'US REIT real estate',
                'UK': 'UK REIT commercial property',
                'EU': 'European REIT real estate',
            }
            mx_query = geo_search.get(geographies[0], 'REIT real estate')
            mx_articles = MarketauxNewsSource.fetch(
                mx_query, self.config.MARKETAUX_KEY,
                days_back=days_back)
            for a in mx_articles:
                a['category'] = 'macro'
                a['query'] = 'MX:macro'
            all_articles.extend(mx_articles)

        return all_articles

    def get_micro_news(self, days_back: int = 30) -> List[Dict]:

        micro_queries = [
            # Direct REIT queries (similar to original)
            f"{self.reit_name} REIT",
            f"{self.reit_name} acquisition divestment",
            f"{self.reit_name} earnings results",
            f"{self.reit_name} distribution DPU",
            f"{self.reit_name} rights issue placement",
            f"{self.reit_name} occupancy rate",
            f"{self.reit_name} property valuation",

            # Sector queries (NEW)
            "Singapore REIT sector performance",
            "S-REIT DPU growth 2025 2026",
        ]

        # Add ticker-based queries
        if self.reit_ticker:
            micro_queries.extend([
                f"{self.reit_ticker} SGX",
                f"{self.reit_ticker} analyst target price",
            ])

        all_articles = []
        for query in micro_queries:
            print(f"  {query}")
            articles = GoogleNewsSource.fetch(query, days_back)
            for a in articles:
                a['category'] = 'micro'
                a['query'] = query
            all_articles.extend(articles)
            time.sleep(0.5)

        # Fetch from financial APIs if configured
        if self.config.ALPHA_VANTAGE_KEY and self.reit_ticker:
            print(f"  [Alpha Vantage] {self.reit_ticker}")
            av_articles = AlphaVantageNewsSource.fetch(
                self.reit_ticker, self.config.ALPHA_VANTAGE_KEY)
            for a in av_articles:
                a['category'] = 'micro'
                a['query'] = f"AV:{self.reit_ticker}"
            all_articles.extend(av_articles)

        if self.config.MARKETAUX_KEY:
            print(f"  [Marketaux] {self.reit_name}")
            mx_articles = MarketauxNewsSource.fetch(
                self.reit_name, self.config.MARKETAUX_KEY,
                days_back=days_back)
            for a in mx_articles:
                a['category'] = 'micro'
                a['query'] = f'MX:{self.reit_name}'
            all_articles.extend(mx_articles)

        return all_articles

    # -----------------------------------------------------------------
    # SENTIMENT ANALYSIS - ENSEMBLE
    # -----------------------------------------------------------------

    def _run_single_model(self, text: str, tokenizer, model, labels) -> Dict[str, float]:
        """Run sentiment analysis through a single model."""
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=512, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        scores = predictions[0].numpy()
        return {
            'positive': float(scores[labels.index('positive')]),
            'negative': float(scores[labels.index('negative')]),
            'neutral': float(scores[labels.index('neutral')]),
            'sentiment': labels[np.argmax(scores)],
            'confidence': float(np.max(scores))
        }

    def analyze_sentiment(self, text: str) -> Dict[str, float]:

        if not text or len(text.strip()) < 5:
            return {
                'positive': 0.33, 'negative': 0.33, 'neutral': 0.34,
                'sentiment': 'neutral', 'confidence': 0.34
            }

        # Primary model
        primary = self._run_single_model(text, self.tokenizer, self.model,
                                          self.primary_labels)

        if self.secondary_model is None:
            return primary

        # Secondary model (ensemble)
        try:
            secondary = self._run_single_model(text, self.secondary_tokenizer,
                                                self.secondary_model,
                                                self.secondary_labels)
        except Exception:
            return primary

        # Weighted average of both models
        w1 = self.config.ENSEMBLE_WEIGHT_PRIMARY
        w2 = 1.0 - w1

        ensemble = {
            'positive': w1 * primary['positive'] + w2 * secondary['positive'],
            'negative': w1 * primary['negative'] + w2 * secondary['negative'],
            'neutral': w1 * primary['neutral'] + w2 * secondary['neutral'],
        }
        ensemble['sentiment'] = max(['positive', 'negative', 'neutral'],
                                     key=lambda k: ensemble[k])
        ensemble['confidence'] = max(ensemble['positive'], ensemble['negative'],
                                      ensemble['neutral'])
        # Also store individual model results for inspection
        ensemble['primary_sentiment'] = primary['sentiment']
        ensemble['secondary_sentiment'] = secondary['sentiment']

        return ensemble

    # -----------------------------------------------------------------
    # ARTICLE PROCESSING PIPELINE
    # -----------------------------------------------------------------

    def process_articles(self, articles: List[Dict],
                          fetch_bodies: bool = False) -> pd.DataFrame:

        print(f"\n📋 Processing pipeline:")
        print(f"   Raw articles: {len(articles)}")

        # Step 1: Deduplicate
        articles = deduplicate_articles(articles)
        print(f"   After deduplication: {len(articles)}")

        # Step 2: Relevance filter
        relevant = [a for a in articles
                     if check_relevance(a, self.reit_name, self.config)]
        filtered_count = len(articles) - len(relevant)
        articles = relevant
        print(f"   After relevance filter: {len(articles)} (removed {filtered_count} off-topic)")

        if not articles:
            print("   ❌ No relevant articles remaining!")
            return pd.DataFrame()

        # Step 3: Optionally fetch article bodies
        if fetch_bodies and HAS_NEWSPAPER:
            print(f"   📰 Fetching article bodies (this may take a while)...")
            for i, article in enumerate(articles):
                body = fetch_article_body(article.get('link', ''))
                article['body'] = body
                if (i + 1) % 20 == 0:
                    print(f"      Fetched {i+1}/{len(articles)} bodies")

        # Step 4: Sentiment analysis
        print(f"\n🤖 Analyzing sentiment...")
        reference_date = datetime.now()
        scored_articles = []

        for i, article in enumerate(articles):
            # Build analysis text - use body if available, else title + description
            if article.get('body'):
                text = f"{article['title']}. {article['body']}"
            else:
                text = f"{article['title']}. {article.get('description', '')}"

            # Run sentiment
            sentiment = self.analyze_sentiment(text)

            # Calculate weights
            recency_weight = calculate_recency_weight(
                article.get('pub_date', ''), reference_date, self.config.RECENCY_DECAY)
            source_quality = get_source_quality_multiplier(
                article.get('source', ''), self.config)

            # Combine into scored article
            scored = {
                **article,
                **sentiment,
                'recency_weight': recency_weight,
                'source_quality': source_quality,
                'weighted_score': (sentiment['positive'] - sentiment['negative'])
                                  * sentiment['confidence']
                                  * recency_weight
                                  * source_quality
            }
            scored_articles.append(scored)

            if (i + 1) % 25 == 0:
                print(f"   Analyzed {i + 1}/{len(articles)} articles")

        print(f"   ✓ Analyzed all {len(articles)} articles")
        return pd.DataFrame(scored_articles)

    # -----------------------------------------------------------------
    # SCORING
    # -----------------------------------------------------------------

    def calculate_category_score(self, df: pd.DataFrame, category: str) -> Dict:

        cat_df = df[df['category'] == category].copy()
        if len(cat_df) == 0:
            return {
                'score': 0.0, 'positive_count': 0, 'negative_count': 0,
                'neutral_count': 0, 'total_articles': 0, 'avg_confidence': 0.0,
                'weighted_score': 0.0
            }

        pos_count = len(cat_df[cat_df['sentiment'] == 'positive'])
        neg_count = len(cat_df[cat_df['sentiment'] == 'negative'])
        neu_count = len(cat_df[cat_df['sentiment'] == 'neutral'])

        # Simple score (same as original for comparison)
        simple_score = ((cat_df['positive'] - cat_df['negative']) * cat_df['confidence']).mean()

        # Weighted score (NEW - accounts for recency and source quality)
        weighted_score = cat_df['weighted_score'].mean()

        return {
            'score': float(simple_score),
            'weighted_score': float(weighted_score),
            'positive_count': pos_count,
            'negative_count': neg_count,
            'neutral_count': neu_count,
            'total_articles': len(cat_df),
            'avg_confidence': float(cat_df['confidence'].mean()),
            'positive_ratio': pos_count / len(cat_df),
            'negative_ratio': neg_count / len(cat_df),
        }

    # -----------------------------------------------------------------
    # VISUALIZATION
    # -----------------------------------------------------------------

    def plot_results(self, df: pd.DataFrame, macro_score: Dict, micro_score: Dict):
        """
        Generate visualization charts.

        IMPROVEMENT: The original had no visualization at all - just printed numbers.
        Charts make it much easier to interpret sentiment patterns.
        """
        if not HAS_PLOTTING or df.empty:
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'Sentiment Analysis: {self.reit_name}', fontsize=14, fontweight='bold')

        # --- Plot 1: Sentiment Distribution by Category ---
        ax1 = axes[0, 0]
        categories = ['macro', 'micro']
        pos_counts = [macro_score['positive_count'], micro_score['positive_count']]
        neg_counts = [macro_score['negative_count'], micro_score['negative_count']]
        neu_counts = [macro_score['neutral_count'], micro_score['neutral_count']]
        x = np.arange(len(categories))
        width = 0.25
        ax1.bar(x - width, pos_counts, width, label='Positive', color='#2ecc71')
        ax1.bar(x, neu_counts, width, label='Neutral', color='#95a5a6')
        ax1.bar(x + width, neg_counts, width, label='Negative', color='#e74c3c')
        ax1.set_xlabel('Category')
        ax1.set_ylabel('Article Count')
        ax1.set_title('Sentiment Distribution')
        ax1.set_xticks(x)
        ax1.set_xticklabels(['Macro', 'Micro'])
        ax1.legend()

        # --- Plot 2: Score Comparison ---
        ax2 = axes[0, 1]
        scores = {
            'Macro\n(Simple)': macro_score['score'],
            'Macro\n(Weighted)': macro_score['weighted_score'],
            'Micro\n(Simple)': micro_score['score'],
            'Micro\n(Weighted)': micro_score['weighted_score'],
        }
        colors = ['#3498db' if v >= 0 else '#e74c3c' for v in scores.values()]
        ax2.bar(scores.keys(), scores.values(), color=colors)
        ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax2.set_title('Sentiment Scores')
        ax2.set_ylabel('Score')

        # --- Plot 3: Confidence Distribution ---
        ax3 = axes[1, 0]
        if 'confidence' in df.columns:
            for cat, color in [('macro', '#3498db'), ('micro', '#e67e22')]:
                cat_conf = df[df['category'] == cat]['confidence']
                if len(cat_conf) > 0:
                    ax3.hist(cat_conf, bins=20, alpha=0.6, label=cat.capitalize(), color=color)
            ax3.set_xlabel('Confidence')
            ax3.set_ylabel('Count')
            ax3.set_title('Model Confidence Distribution')
            ax3.legend()

        # --- Plot 4: Sentiment Over Time ---
        ax4 = axes[1, 1]
        try:
            df_time = df.copy()
            for fmt in ['%a, %d %b %Y %H:%M:%S %Z', '%a, %d %b %Y %H:%M:%S GMT',
                        '%Y-%m-%dT%H:%M:%SZ', '%Y-%m-%d']:
                try:
                    df_time['parsed_date'] = pd.to_datetime(df_time['pub_date'], format=fmt)
                    break
                except (ValueError, TypeError):
                    continue
            else:
                df_time['parsed_date'] = pd.to_datetime(df_time['pub_date'], errors='coerce')

            df_time = df_time.dropna(subset=['parsed_date'])
            if len(df_time) > 0:
                daily = df_time.set_index('parsed_date').resample('D')['weighted_score'].mean()
                daily = daily.dropna()
                if len(daily) > 0:
                    ax4.plot(daily.index, daily.values, marker='o', linewidth=1.5, color='#2c3e50')
                    ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
                    ax4.fill_between(daily.index, daily.values, 0,
                                     where=(daily.values >= 0), alpha=0.3, color='green')
                    ax4.fill_between(daily.index, daily.values, 0,
                                     where=(daily.values < 0), alpha=0.3, color='red')
                    ax4.set_xlabel('Date')
                    ax4.set_ylabel('Weighted Score')
                    ax4.set_title('Daily Sentiment Trend')
                    ax4.tick_params(axis='x', rotation=45)
        except Exception as e:
            ax4.text(0.5, 0.5, f'Could not parse dates\n{e}',
                     ha='center', va='center', transform=ax4.transAxes)

        plt.tight_layout()
        plt.savefig('sentiment_analysis_charts.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ Charts saved to sentiment_analysis_charts.png")

    # -----------------------------------------------------------------
    # MAIN REPORT GENERATION
    # -----------------------------------------------------------------

    def generate_report(self, days_back: int = 30,
                         fetch_bodies: bool = False) -> Optional[Dict]:

        print(f"\n{'='*60}")
        print(f"IMPROVED Sentiment Analysis: {self.reit_name}")
        print(f"Period: Last {days_back} days")
        print(f"Models: {self.config.PRIMARY_MODEL}" +
              (f" + {self.config.SECONDARY_MODEL}" if self.secondary_model else ""))
        print(f"{'='*60}")

        # Fetch news
        print(f"\n📰 Fetching MACRO news...")
        macro_articles = self.get_macro_news(days_back)

        print(f"\n📰 Fetching MICRO news...")
        micro_articles = self.get_micro_news(days_back)

        all_articles = macro_articles + micro_articles
        if not all_articles:
            print("❌ No articles found!")
            return None

        print(f"\n✓ Raw articles found: {len(all_articles)}")
        print(f"  Macro: {len(macro_articles)} | Micro: {len(micro_articles)}")

        # Process through full pipeline
        df = self.process_articles(all_articles, fetch_bodies=fetch_bodies)
        if df.empty:
            print("❌ No articles passed filtering!")
            return None

        # Calculate scores
        macro_score = self.calculate_category_score(df, 'macro')
        micro_score = self.calculate_category_score(df, 'micro')

        # Combined score (weighted by config)
        w_macro = self.config.MACRO_WEIGHT
        w_micro = self.config.MICRO_WEIGHT
        combined_simple = w_macro * macro_score['score'] + w_micro * micro_score['score']
        combined_weighted = (w_macro * macro_score['weighted_score'] +
                              w_micro * micro_score['weighted_score'])

        # Print results
        print(f"\n{'='*60}")
        print(f"📊 MACRO SENTIMENT (weight: {w_macro:.0%})")
        print(f"{'='*60}")
        print(f"Simple Score:   {macro_score['score']:.4f}")
        print(f"Weighted Score: {macro_score['weighted_score']:.4f}")
        print(f"Positive: {macro_score['positive_count']} | "
              f"Negative: {macro_score['negative_count']} | "
              f"Neutral: {macro_score['neutral_count']}")
        print(f"Total articles: {macro_score['total_articles']}")
        print(f"Avg Confidence: {macro_score['avg_confidence']:.2%}")

        print(f"\n{'='*60}")
        print(f"📊 MICRO SENTIMENT (weight: {w_micro:.0%})")
        print(f"{'='*60}")
        print(f"Simple Score:   {micro_score['score']:.4f}")
        print(f"Weighted Score: {micro_score['weighted_score']:.4f}")
        print(f"Positive: {micro_score['positive_count']} | "
              f"Negative: {micro_score['negative_count']} | "
              f"Neutral: {micro_score['neutral_count']}")
        print(f"Total articles: {micro_score['total_articles']}")
        print(f"Avg Confidence: {micro_score['avg_confidence']:.2%}")

        print(f"\n{'='*60}")
        print(f"📈 COMBINED SCORE")
        print(f"{'='*60}")
        print(f"Simple Combined:   {combined_simple:.4f}")
        print(f"Weighted Combined: {combined_weighted:.4f}")
        if combined_weighted > 0.3:
            signal = "STRONGLY BULLISH 🟢🟢"
        elif combined_weighted > 0.15:
            signal = "BULLISH 🟢"
        elif combined_weighted > 0.05:
            signal = "SLIGHTLY BULLISH 🟢"
        elif combined_weighted > -0.05:
            signal = "NEUTRAL 🟡"
        elif combined_weighted > -0.15:
            signal = "SLIGHTLY BEARISH 🔴"
        elif combined_weighted > -0.3:
            signal = "BEARISH 🔴"
        else:
            signal = "STRONGLY BEARISH 🔴🔴"
        print(f"Signal: {signal}")

        # Generate visualization
        self.plot_results(df, macro_score, micro_score)

        # Build report dict
        report = {
            'reit_name': self.reit_name,
            'reit_ticker': self.reit_ticker,
            'analysis_date': datetime.now().isoformat(),
            'period_days': days_back,
            'macro': macro_score,
            'micro': micro_score,
            'combined_simple': combined_simple,
            'combined_weighted': combined_weighted,
            'signal': signal,
            'articles_df': df,
            'config': {
                'primary_model': self.config.PRIMARY_MODEL,
                'secondary_model': self.config.SECONDARY_MODEL,
                'macro_weight': self.config.MACRO_WEIGHT,
                'micro_weight': self.config.MICRO_WEIGHT,
            },
            'summary': {
                'total_articles': len(df),
                'macro_articles': macro_score['total_articles'],
                'micro_articles': micro_score['total_articles'],
            }
        }

        return report

## RUN ANALYSIS

In [ ]:
config = AnalysisConfig()

# ── API Keys (replace with your own keys) ──
config.ALPHA_VANTAGE_KEY = "FY4C9KVQUAHLM3QC"
config.NEWSDATA_KEY = "pub_8008be31a399490bbefa10de2181a857"
config.MARKETAUX_KEY = "YOUR_MARKETAUX_API_KEY"  # Get free key at https://www.marketaux.com

# ── Geography: set based on where REIT properties are located ──
# Options: "SG", "US", "UK", "EU", "GLOBAL"
# BUOU (Frasers L&C Trust) has SG + AU + EU exposure
config.GEOGRAPHIES = ["SG", "EU"]

analyzer = ImprovedREITSentimentAnalyzer(
    reit_name="Frasers Logistics & Comm Trust",
    reit_ticker="BUOU",
    config=config
)

# --- Generate Report ---
# Set fetch_bodies=True for richer analysis (slower, requires newspaper4k)
report = analyzer.generate_report(days_back=30, fetch_bodies=False)

## POST-ANALYSIS

In [ ]:
if report:
    # Save to CSV
    filename = f"reit_sentiment_improved_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    report['articles_df'].to_csv(filename, index=False)
    print(f"\n✓ Saved to {filename}")

    # Top positive articles (filtered and deduplicated)
    print(f"\n{'='*60}")
    print("🟢 TOP 5 POSITIVE ARTICLES")
    print(f"{'='*60}")
    top_pos = report['articles_df'].nlargest(5, 'positive')[
        ['title', 'positive', 'confidence', 'category', 'source', 'weighted_score']
    ]
    for idx, row in top_pos.iterrows():
        print(f"\n{row['title'][:80]}...")
        print(f"  Positive: {row['positive']:.3f} | Confidence: {row['confidence']:.3f} | "
              f"Weighted: {row['weighted_score']:.3f}")
        print(f"  Category: {row['category']} | Source: {row['source']}")

    # Top negative articles
    print(f"\n{'='*60}")
    print("🔴 TOP 5 NEGATIVE ARTICLES")
    print(f"{'='*60}")
    top_neg = report['articles_df'].nlargest(5, 'negative')[
        ['title', 'negative', 'confidence', 'category', 'source', 'weighted_score']
    ]
    for idx, row in top_neg.iterrows():
        print(f"\n{row['title'][:80]}...")
        print(f"  Negative: {row['negative']:.3f} | Confidence: {row['confidence']:.3f} | "
              f"Weighted: {row['weighted_score']:.3f}")
        print(f"  Category: {row['category']} | Source: {row['source']}")

    # Source distribution
    print(f"\n{'='*60}")
    print("📰 TOP SOURCES")
    print(f"{'='*60}")
    source_counts = report['articles_df']['source'].value_counts().head(10)
    for source, count in source_counts.items():
        quality = get_source_quality_multiplier(source, config)
        tier = "⭐" if quality > 1.1 else ""
        print(f"  {source}: {count} articles {tier}")

    # Final summary
    print(f"\n{'='*60}")
    print("📊 FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"REIT: {report['reit_name']} ({report['reit_ticker']})")
    print(f"Period: Last {report['period_days']} days")
    print(f"Articles Analyzed: {report['summary']['total_articles']}")
    print(f"  Macro: {report['summary']['macro_articles']} | "
          f"Micro: {report['summary']['micro_articles']}")
    print(f"Macro Score (weighted): {report['macro']['weighted_score']:.4f}")
    print(f"Micro Score (weighted): {report['micro']['weighted_score']:.4f}")
    print(f"Combined Weighted Score: {report['combined_weighted']:.4f}")
    print(f"Signal: {report['signal']}")
else:
    print("❌ No report generated")